# Demo v0.4 — все режимы расчёта

Собираем «коробку с инструментами» проектировщика: пять режимов на одном движке `compute_tep_for_kit`.

| Режим | Функция | Когда |
|---|---|---|
| Максимальный КИТ | `solve_max_kit` | «На что максимум можно рассчитывать?» |
| КИТ с резервом | `solve_max_kit_with_reserve` | «Хочу, чтобы остался запас 5–10 тыс. м² на манёвр» |
| КИТ при заданном ЗНОП | `solve_max_kit_with_znop` | «Хочу 6 м²/чел зелени, при каком КИТ это совместимо?» |
| Проверка КИТ | `verify_kit` | «У меня уже есть КИТ, что получается?» |
| Сравнение | `compare_scenarios` | «Сравним варианты бок о бок» |

Плюс **ВПП** как самостоятельная сущность: `BuiltInArea(area_m2, vri_code)` — встроенно-пристроенные помещения со своими нормами парковок и озеленения по ВРИ.

In [1]:
import pathlib

from urban_model import (
    solve_max_kit,
    solve_max_kit_with_reserve,
    solve_max_kit_with_znop,
    verify_kit,
    compare_scenarios,
)
from urban_model.modes.compare import run_scenarios
from urban_model.models import (
    Site, CalculationOptions, ParkingConfig, BuiltInArea, Scenario,
)
from urban_model.normatives import load_normatives
from urban_model.export import to_xlsx

norms = load_normatives('spb')
site = Site(area_m2=100_000, name='Квартал 10 га')

## 1. Максимальный КИТ

In [2]:
opts_base = CalculationOptions(floors=15, planning_doc=True)
r1 = solve_max_kit(site, opts_base, norms)
print(r1.summary())

Профиль: spb
КИТ:                     1.200 (норм. макс 2.5)
Площадь квартир:         89,985 м²
Население:               3,214 чел
Плотность:               321.4 чел/га [ok]
ДОО (мест):              требуется 196.03951590401783 → принято 200
СОШ (мест):              требуется 385.6515066964285 → принято 390
ЗНОП (м²/чел):           0 → итого 0 м²
Парковки (м/м):
  всего требуется        1125
  открытые               141 м/м, 2,926 м²
  подземные              984 м/м (без поверхностной площади)
Баланс:                  OK (+16,180 м²)
Ограничивающий фактор:   ЗУ жилой застройки (41,220 м², 41.2% квартала)
  ⚠ СОШ: расчётная вместимость [390] < нормативного минимума 550 мест — стандартная отдельно стоящая СОШ невозможна, нужна стоянка-спутник или ВПП-школа (учтётся в v0.2).


## 2. КИТ с резервом 10 000 м²

Хочу, чтобы после всех нагрузок осталось **минимум 10 тыс. м²** свободной площади на квартале — на дополнительное благоустройство, спортивную площадку, площадь.

In [3]:
r2 = solve_max_kit_with_reserve(site, target_surplus_m2=10_000, options=opts_base, norms=norms)
print(r2.summary())

Профиль: spb
КИТ:                     1.200 (норм. макс 2.5)
Площадь квартир:         89,985 м²
Население:               3,214 чел
Плотность:               321.4 чел/га [ok]
ДОО (мест):              требуется 196.03951590401783 → принято 200
СОШ (мест):              требуется 385.6515066964285 → принято 390
ЗНОП (м²/чел):           0 → итого 0 м²
Парковки (м/м):
  всего требуется        1125
  открытые               141 м/м, 2,926 м²
  подземные              984 м/м (без поверхностной площади)
Баланс:                  OK (+16,180 м²)
Ограничивающий фактор:   подбор по резерву ≥ 10,000 м²; ЗУ жилой застройки (41,220 м², 41.2% квартала)
  ⚠ СОШ: расчётная вместимость [390] < нормативного минимума 550 мест — стандартная отдельно стоящая СОШ невозможна, нужна стоянка-спутник или ВПП-школа (учтётся в v0.2).


## 3. Принудительный ЗНОП = 6 м²/чел

Хочу для жителей выше нормативных 0/3/4 м²/чел зелёных насаждений общего пользования. При каком КИТ это совместимо?

Поле `znop_per_person.status = manual` — это явная пользовательская правка.

In [4]:
r3 = solve_max_kit_with_znop(site, target_znop_per_person=6, options=opts_base, norms=norms)
print(r3.summary())
print(f'\nЗНОП-поле: status = {r3.znop_per_person.status.value}, formula = {r3.znop_per_person.formula!r}')

Профиль: spb
КИТ:                     1.154 (норм. макс 2.5)
Площадь квартир:         86,514 м²
Население:               3,090 чел
Плотность:               309.0 чел/га [ok]
ДОО (мест):              требуется 188.47621372767856 → принято 190
СОШ (мест):              требуется 370.77287946428567 → принято 380
ЗНОП (м²/чел):           6 → итого 18,539 м²
Парковки (м/м):
  всего требуется        1082
  открытые               136 м/м, 2,822 м²
  подземные              946 м/м (без поверхностной площади)
Баланс:                  OK (+23 м²)
Ограничивающий фактор:   ЗНОП зафиксирован вручную = 6 м²/чел; ЗУ жилой застройки (39,638 м², 39.6% квартала)
  ⚠ СОШ: расчётная вместимость [380] < нормативного минимума 550 мест — стандартная отдельно стоящая СОШ невозможна, нужна стоянка-спутник или ВПП-школа (учтётся в v0.2).

ЗНОП-поле: status = manual, formula = 'override = 6'


## 4. ВПП — встроенно-пристроенные помещения

Добавляем в проект **5 000 м² торгового центра** (ВРИ 4.4) на 1-2 этажах жилого корпуса. Программа:
- вычитает площадь ВПП из GFA при расчёте квартир;
- добавляет парковки ВПП по нормативу ВРИ-4.4 (1 м/м на 50 м²);
- добавляет озеленение ВПП (15 м²/100 м²).

In [5]:
opts_vpp = CalculationOptions(
    floors=15, planning_doc=True,
    built_in=BuiltInArea(area_m2=5_000, vri_code='4.4', label='Торговый центр'),
)
r4 = solve_max_kit(site, opts_vpp, norms)
print(r4.summary())
print(f'\nИсточник нормы парковки 4.4: {r4.built_in_parking_places.source}')

Профиль: spb
КИТ:                     1.250 (норм. макс 2.5)
Площадь квартир:         89,971 м²
ВПП:                     5,000 м² (ВРИ 4.4), парковки +100 м/м
Население:               3,213 чел
Плотность:               321.3 чел/га [ok]
ДОО (мест):              требуется 196.00760323660717 → принято 200
СОШ (мест):              требуется 385.5887276785715 → принято 390
ЗНОП (м²/чел):           0 → итого 0 м²
Парковки (м/м):
  всего требуется        1225
  открытые               154 м/м, 3,196 м²
  подземные              1071 м/м (без поверхностной площади)
Баланс:                  OK (+14,434 м²)
Ограничивающий фактор:   ЗУ жилой застройки (42,966 м², 43.0% квартала)
  ⚠ СОШ: расчётная вместимость [390] < нормативного минимума 550 мест — стандартная отдельно стоящая СОШ невозможна, нужна стоянка-спутник или ВПП-школа (учтётся в v0.2).

Источник нормы парковки 4.4: ПЗЗ СПб — ориентировочно, требует уточнения


## 5. Сравнение всех 4 сценариев в одной таблице

Видны: разница в КИТ, нагрузке на парковки, балансе территории, ограничивающем факторе.

In [6]:
import pandas as pd

# Соберём пары (имя, TEPResult) в правильном порядке
pairs = [
    ('Макс. КИТ', r1),
    ('С резервом 10k', r2),
    ('ЗНОП=6 м²/чел', r3),
    ('С ВПП 5000 м²', r4),
]

from urban_model.export import results_to_dataframe
df = results_to_dataframe(pairs)
rows = [
    'КИТ', 'Площадь квартир, м²', 'Население, чел',
    'ЗНОП, м²/чел', 'ЗНОП итого, м²',
    'ДОО мест (принято)', 'СОШ мест (принято)',
    'Парковки всего, м/м', 'Откр. парковки, м/м',
    'Баланс территории, м²', 'Ограничивающий фактор',
]
df.loc[rows]

сценарий,Макс. КИТ,С резервом 10k,ЗНОП=6 м²/чел,С ВПП 5000 м²
показатель,,,,
КИТ,1.2,1.2,1.15,1.25
"Площадь квартир, м²",89985.35,89985.35,86513.67,89970.7
"Население, чел",3213.76,3213.76,3089.77,3213.24
"ЗНОП, м²/чел",0,0,6,0
"ЗНОП итого, м²",0.0,0.0,18538.64,0.0
ДОО мест (принято),200,200,190,200
СОШ мест (принято),390,390,380,390
"Парковки всего, м/м",1125,1125,1082,1225
"Откр. парковки, м/м",141,141,136,154


## 6. Экспорт сводного отчёта в Excel

In [7]:
out_dir = pathlib.Path('../output')
out_dir.mkdir(exist_ok=True)
path = to_xlsx(pairs, out_dir / 'tep_v0_4.xlsx')
print(f'Записан: {path.resolve()}')
print(f'Размер:  {path.stat().st_size:,} байт')

Записан: D:\Github\my_urban_model\output\tep_v0_4.xlsx
Размер:  13,536 байт


## Итог

v0.2 закрыт полностью:
- 5 режимов расчёта на одном движке
- ВПП с собственными парковками и озеленением по ВРИ
- Парковочные сценарии (open / multilevel / underground)
- Аудит-трейл (status / source / formula) на каждом поле
- Excel-экспорт с цветовой маркировкой статусов

Дальше — v0.3: расширение соцобъектов (поликлиники, ФОК, культура), парковки соцобъектов на стоянках-спутниках.